# Singular-Jacobian diagnostics

This notebook inspects detJ=0 and near-zero response points from `*_detJ_singular.csv.gz`.

The official extraction still uses the `_all.csv.gz` file.  
This notebook is diagnostic only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from YAnalysis_singular import SingularTransitionData, load_singular_csv, suggest_bins
from YPlot_singular import (
    plot_hist2d_count,
    plot_hist2d_weighted,
    plot_sign_scatter,
    plot_dominant_sign_map,
    plot_mixed_sign_fraction,
    plot_histogram,
    add_observable_center,
    save_figure_pair,
)

In [ ]:
# Replace this placeholder with a P-ADO singular diagnostic CSV.
CSV_PATH = Path(r"path\to\file_name_detJ_singular.csv.gz")

# Plot controls. Adjust these manually if needed.
PADO_BINS = 300
AT_SIGMA_BINS = (360, 250)
SIGN_BINS = 220
DENSITY_REL_CORE = 1e-3
SCATTER_SAMPLE = 300_000

# Display ranges.
# Use None for automatic data range.
PADO_RANGE = None
AT_SIGMA_RANGE = ((-90, 90), (0, 1))

# Figure output folder under the current final project.
FIG_DIR = Path("outputs") / "singular_figures"

# Experimental center used for the current analysis.
EXP_ADO = 0.390
EXP_ADO_ERR = 0.050
EXP_P = -0.120
EXP_P_ERR_LEFT = 0.120
EXP_P_ERR_RIGHT = 0.110

## Select singular CSV

Set `CSV_PATH` in the configuration cell to a generated `*_detJ_singular.csv` or `*_detJ_singular.csv.gz` file.

In [ ]:
singular_path = CSV_PATH.expanduser()

if not singular_path.is_file():
    raise FileNotFoundError(f"Singular file does not exist: {singular_path}")

print("Selected singular file:")
print(singular_path)
print(f"File size: {singular_path.stat().st_size / (1024**2):.2f} MiB")

In [ ]:
data = SingularTransitionData(singular_path)
df = data.df

print(f"Rows: {len(df):,}")
print(f"File: {data.path}")
print(f"Columns ({len(df.columns)}):")
for column in df.columns:
    print(f"  {column}")
print(f"Suggested square-bin count: {suggest_bins(len(df))}")

display(data.summary())
display(data.class_counts())
display(data.sign_counts())
display(data.density_quantiles())

In [ ]:
df_all = df
df_core = data.filter_by_density_rel(DENSITY_REL_CORE)
df_zero = data.filter_by_class(["zero"])
df_near = data.filter_by_class(["near_zero"])

print(f"all:       {len(df_all):,}")
print(f"core:      {len(df_core):,}  (density rel >= {DENSITY_REL_CORE:g})")
print(f"zero:      {len(df_zero):,}")
print(f"near_zero: {len(df_near):,}")
print(f"Figures will be saved under: {FIG_DIR}")

## P-ADO diagnostics

In [ ]:
fig, ax, artist = plot_sign_scatter(
    df_all, "ado", "p", sample=SCATTER_SAMPLE, s=0.5, alpha=0.12,
    title="P-ADO singular points by signed detJ",
)
add_observable_center(
    ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
    p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT,
)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_sign_scatter")
plt.show()

In [ ]:
fig, ax, artist = plot_hist2d_count(
    df_all, "ado", "p", bins=PADO_BINS, range=PADO_RANGE,
    title="P-ADO singular count density",
)
add_observable_center(ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
                      p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_count_hist2d")
plt.show()

In [ ]:
fig, ax, artist = plot_hist2d_weighted(
    df_all, "ado", "p", weight_col="observable_density_rel_to_max",
    bins=PADO_BINS, range=PADO_RANGE,
    title="P-ADO singular observable-density-weighted map",
)
add_observable_center(ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
                      p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_weighted_hist2d")
plt.show()

In [ ]:
fig, ax, artist = plot_dominant_sign_map(
    df_all, "ado", "p", bins=SIGN_BINS, range=PADO_RANGE,
    title="P-ADO dominant signed-detJ sign",
)
add_observable_center(ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
                      p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_dominant_sign")
plt.show()

In [ ]:
fig, ax, artist = plot_mixed_sign_fraction(
    df_all, "ado", "p", bins=SIGN_BINS, range=PADO_RANGE,
    title="P-ADO mixed signed-detJ fraction",
)
add_observable_center(ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
                      p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_mixed_sign_fraction")
plt.show()

In [ ]:
fig, ax, artist = plot_sign_scatter(
    df_core, "ado", "p", sample=SCATTER_SAMPLE, s=0.7, alpha=0.18,
    title=f"P-ADO core singular points, density rel >= {DENSITY_REL_CORE:g}",
)
add_observable_center(ax, ado=EXP_ADO, p=EXP_P, ado_err=EXP_ADO_ERR,
                      p_err_left=EXP_P_ERR_LEFT, p_err_right=EXP_P_ERR_RIGHT)
# save_figure_pair(fig, FIG_DIR, "singular_P-ADO_core_sign_scatter")
plt.show()

## AT-sigma/I diagnostics

In [ ]:
fig, ax, artist = plot_sign_scatter(
    df_all, "ArcTan[delta](AT)", "sigma/I", sample=SCATTER_SAMPLE,
    sign_col="detJ_sign(ArcTan[delta](AT),sigma/I)",
    s=0.5, alpha=0.12, title="AT-sigma/I singular points by signed detJ",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_sign_scatter")
plt.show()

In [ ]:
fig, ax, artist = plot_hist2d_count(
    df_all, "ArcTan[delta](AT)", "sigma/I", bins=AT_SIGMA_BINS,
    range=AT_SIGMA_RANGE, title="AT-sigma/I singular count density",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_count_hist2d")
plt.show()

In [ ]:
fig, ax, artist = plot_hist2d_weighted(
    df_all, "ArcTan[delta](AT)", "sigma/I",
    weight_col="observable_density_rel_to_max", bins=AT_SIGMA_BINS,
    range=AT_SIGMA_RANGE,
    title="AT-sigma/I singular observable-density-weighted map",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_weighted_hist2d")
plt.show()

In [ ]:
fig, ax, artist = plot_dominant_sign_map(
    df_all, "ArcTan[delta](AT)", "sigma/I", bins=SIGN_BINS,
    sign_col="detJ_sign(ArcTan[delta](AT),sigma/I)",
    range=AT_SIGMA_RANGE, title="AT-sigma/I dominant signed-detJ sign",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_dominant_sign")
plt.show()

In [ ]:
fig, ax, artist = plot_mixed_sign_fraction(
    df_all, "ArcTan[delta](AT)", "sigma/I", bins=SIGN_BINS,
    sign_col="detJ_sign(ArcTan[delta](AT),sigma/I)",
    range=AT_SIGMA_RANGE, title="AT-sigma/I mixed signed-detJ fraction",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_mixed_sign_fraction")
plt.show()

In [ ]:
fig, ax, artist = plot_sign_scatter(
    df_core, "ArcTan[delta](AT)", "sigma/I", sample=SCATTER_SAMPLE,
    sign_col="detJ_sign(ArcTan[delta](AT),sigma/I)",
    s=0.7, alpha=0.18,
    title=f"AT-sigma/I core singular points, density rel >= {DENSITY_REL_CORE:g}",
)
# save_figure_pair(fig, FIG_DIR, "singular_AT_sigma_core_sign_scatter")
plt.show()

## One-dimensional distributions

In [ ]:
histogram_specs = [
    ("ArcTan[delta](AT)", "singular_hist_AT", "Singular-point AT distribution"),
    ("sigma/I", "singular_hist_sigma", "Singular-point sigma/I distribution"),
    ("observable_density_rel_to_max", "singular_hist_density_rel", "Singular-point relative observable density"),
    ("detJ_signed(delta,sigma/I)", "singular_hist_detJ_signed", "Singular-point signed detJ distribution"),
]

for column, stem, title in histogram_specs:
    if column not in df_all.columns:
        print(f"Skipping unavailable histogram column: {column}")
        continue
    fig, ax, artist = plot_histogram(df_all, column, bins=100, logy=True, title=title)
    # save_figure_pair(fig, FIG_DIR, stem)
    plt.show()

## Interpretation checklist

- Are singular points concentrated at very low \(\sigma/I\)?
- Are singular points concentrated in observable-density tails?
- Do zero and near-zero classes occupy different regions?
- Do positive and negative signed detJ points form separated branches?
- In P-ADO space, do positive and negative signed detJ points overlap in the same region?
- Does the core-density subset overlap the experimental P-ADO center?
- Does the singular region suggest weak response, local folding, or branch overlap?
- The official extraction still uses the `_all.csv.gz` file; this notebook is diagnostic only.